In [2]:
# Cell 1: تثبيت المكتبات المطلوبة لعملية التعلم الآلي ومعالجة النصوص
%pip install scikit-learn nltk pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Cell 2: استدعاء جميع المكتبات اللي هنستخدمها في المشروع (Logistic Regression)
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pickle

nltk.download('stopwords', quiet=True)
print("Libraries loaded successfully ")

Libraries loaded successfully 


In [4]:
# Cell 3: تحميل الداتا وتنظيف النصوص (إزالة الروابط والأرقام والكلمات المشتركة)
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words and len(word) > 1]
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(word) for word in tokens]
    return ' '.join(tokens)

print("Loading dataset...")
df = pd.read_csv("spam.csv", encoding='latin-1')
df = df[['v1', 'v2']]
df.columns = ['label', 'message']
df['label_encoded'] = df['label'].map({'ham': 0, 'spam': 1})
df['cleaned'] = df['message'].apply(preprocess_text)

print(f"Dataset loaded: {df.shape[0]} messages")
print(f"Spam: {(df['label'] == 'spam').sum()}")
print(f"Ham:  {(df['label'] == 'ham').sum()}")

Loading dataset...
Dataset loaded: 5572 messages
Spam: 747
Ham:  4825


In [5]:
# Cell 4: تقسيم الداتا، تدريب موديل Logistic Regression، واختبار دقته
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned'], df['label_encoded'],
    test_size=0.2, random_state=42, stratify=df['label_encoded']
)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)

y_pred = lr_model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print(f"ACCURACY: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

ACCURACY: 96.86%

Classification Report:
              precision    recall  f1-score   support

         Ham       0.97      1.00      0.98       966
        Spam       0.98      0.78      0.87       149

    accuracy                           0.97      1115
   macro avg       0.97      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115



In [6]:
# Cell 5: حفظ موديل Logistic Regression والـ Vectorizer كملفات pkl لاستخدامهم لاحقاً في الـ GUI
with open("lr_model.pkl", 'wb') as f:
    pickle.dump(lr_model, f)
print("lr_model.pkl saved! ")

with open("lr_vectorizer.pkl", 'wb') as f:
    pickle.dump(vectorizer, f)
print("lr_vectorizer.pkl saved! ")

lr_model.pkl saved! 
lr_vectorizer.pkl saved! 


In [7]:
# Cell 6: تجربة الموديل يدوياً بإدخال أي رسالة لمعرفة هل هي سبام ولا لا
import pickle

# 1. تحميل الموديل والـ Vectorizer اللي حفظناهم في الخطوة السابقة
with open("lr_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)
with open("lr_vectorizer.pkl", "rb") as f:
    loaded_vectorizer = pickle.load(f)

# 2. عمل دالة تأخذ الرسالة وترجع النتيجة
def test_message(text):
    cleaned = preprocess_text(text) # بنستخدم دالة التنظيف من الخلية 3
    vec = loaded_vectorizer.transform([cleaned])
    prediction = loaded_model.predict(vec)[0]
    if prediction == 1:
        return " Spam"
    else:
        return " Not Spam"

# 3. جربي تغيير الرسالة دي لأي رسالة عايزتها
my_msg = "Congratulations! You've won a FREE $1000 gift card."
print(f"Message: {my_msg}")
print(f"Result:  {test_message(my_msg)}")

Message: Congratulations! You've won a FREE $1000 gift card.
Result:   Not Spam


In [8]:
my_msg1 = "Win a FREE vacation to Dubai! Just enter your details: wintrip-now.com"
print(f"Message: {my_msg1}")
print(f"Result:  {test_message(my_msg1)}")

Message: Win a FREE vacation to Dubai! Just enter your details: wintrip-now.com
Result:   Spam
